# Colab 03 - Evaluation / Ablation

Run final metrics after Qdrant Cloud, OpenRouter, and `MyDrive/KS_Project_2/data` are configured.

Required Colab Secrets:
- `OPENROUTER_API_KEY`
- `QDRANT_URL`
- `QDRANT_API_KEY`

## 1. Clone / update project

This cell always lands in `/content/project-ks2` and verifies the repository layout.

In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
from pathlib import Path

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git pull --ff-only || true

root = Path.cwd()
print("cwd:", root)
print("pyproject exists:", Path("pyproject.toml").exists())
print("medical_rag exists:", Path("src/medical_rag").exists())
assert Path("pyproject.toml").exists(), "Wrong folder: pyproject.toml not found"
assert Path("src/medical_rag").exists(), "Wrong folder: src/medical_rag not found"


## 2. Install dependencies

Editable install keeps local source changes active in Colab.

In [ ]:
!python -m pip install -U pip
!python -m pip install -e ".[qdrant,agent,eval]"
!python -m pip install -U requests huggingface_hub
!python -c "import medical_rag; print('medical_rag import OK')"


## 3. Configure secrets and model defaults

Secrets are read from Colab Secrets. BioMedBERT uses the correct public large checkpoint.

In [ ]:
import os

try:
    from google.colab import userdata
    for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY", "HF_TOKEN"]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("BIOMEDBERT_MODEL", "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract")
os.environ.setdefault("BIOMEDBERT_DIM", "1024")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("BIOMEDBERT_MODEL:", os.environ.get("BIOMEDBERT_MODEL"))
print("BIOMEDBERT_DIM:", os.environ.get("BIOMEDBERT_DIM"))

assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY in Colab Secrets"
assert os.environ.get("QDRANT_URL"), "Missing QDRANT_URL in Colab Secrets"
assert os.environ.get("QDRANT_API_KEY"), "Missing QDRANT_API_KEY in Colab Secrets"


## Data sync from Google Drive

Upload local data to `/content/drive/MyDrive/KS_Project_2/data`. This cell copies it into `/content/project-ks2/data` and fails early if it is missing.

In [ ]:
from pathlib import Path

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/KS_Project_2/data")
LOCAL_DATA_DIR = Path("/content/project-ks2/data")

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped/unavailable:", exc)

print("Drive data dir:", DRIVE_DATA_DIR)
print("Drive data exists:", DRIVE_DATA_DIR.exists())
assert DRIVE_DATA_DIR.exists(), f"Missing {DRIVE_DATA_DIR}. Upload data to MyDrive/KS_Project_2/data first."

!mkdir -p "{LOCAL_DATA_DIR}"
!rsync -ah --delete --progress "{DRIVE_DATA_DIR}/" "{LOCAL_DATA_DIR}/"

files = [p for p in LOCAL_DATA_DIR.rglob("*") if p.is_file()]
print("Local data dir:", LOCAL_DATA_DIR)
print("data file count:", len(files))
for p in files[:50]:
    print(" -", p)
assert files, "Local data folder is empty after sync"


## Verify data availability

Qdrant indexing/evaluation needs `data/` to exist and contain dataset files.

In [ ]:
from pathlib import Path

data_dir = Path("data")
print("cwd:", Path.cwd())
print("data exists:", data_dir.exists())
assert data_dir.exists(), "Missing data/. Run the Drive data sync cell first."

files = [p for p in data_dir.rglob("*") if p.is_file()]
print("file count:", len(files))
for p in files[:80]:
    print(" -", p)
assert files, "data/ exists but contains no files"

patterns = ["*.json", "*.jsonl", "*.csv", "*.parquet", "*.jpg", "*.jpeg", "*.png", "*.pkl", "*.joblib"]
for pat in patterns:
    matches = list(data_dir.rglob(pat))
    print(f"data/**/*{pat[1:]}", len(matches))


## Provider diagnostics

In [ ]:
!python -m medical_rag test-openrouter
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


## Baseline advanced metrics

In [ ]:
!mkdir -p outputs/benchmark
!python scripts/colab_workflow.py eval-baseline   --eval-file data/eval_cases.json   --output-file outputs/benchmark/baseline_advanced.json


## Agent metrics with Qdrant Cloud + OpenRouter

In [ ]:
!mkdir -p outputs/benchmark
!python scripts/colab_workflow.py eval-agent   --eval-file data/eval_cases.json   --output-file outputs/benchmark/agent_openrouter.json   --use-qdrant


## Ablation report

In [ ]:
!mkdir -p outputs/ablation
!python scripts/colab_workflow.py ablation   --eval-file data/eval_cases.json   --output-dir outputs/ablation


## Save outputs to Drive

In [ ]:
from pathlib import Path

OUT_DRIVE = Path("/content/drive/MyDrive/KS_Project_2/outputs")
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
!rsync -ah --delete outputs/ "{OUT_DRIVE}/"
print("Saved outputs to", OUT_DRIVE)
